<a href="https://colab.research.google.com/github/Jayan-V/Data-Analysis/blob/main/Predictive_Analysis_of_Crop_Yield_Based_on_Nitrogen_and_Rainfall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
data = pd.read_csv('/content/Crop_recommendation.csv')

## Filter Data for Rice Crop



In [ ]:
df_rice = data[data['label'] == 'rice'].copy()
print("Shape of df_rice:", df_rice.shape)
df_rice.head()

##Yield Score for Rice


In [ ]:
df_rice['Yield_Score'] = (
    0.5 * df_rice['N'] +
    0.1 * df_rice['P'] +
    0.1 * df_rice['K'] +
    0.2 * df_rice['temperature'] +
    0.1 * df_rice['humidity'] +
    0.3 * df_rice['rainfall'] -
    0.4 * (df_rice['ph'] - 6.5)**2
)

print("DataFrame df_rice with new 'Yield_Score' column:")
df_rice.head()

## Normalize Yield Score for Rice



In [ ]:
min_yield_score = df_rice['Yield_Score'].min()
max_yield_score = df_rice['Yield_Score'].max()

df_rice['Yield_Score_Normalized'] = ((df_rice['Yield_Score'] - min_yield_score) / (max_yield_score - min_yield_score)) * 100

print("DataFrame df_rice with new 'Yield_Score_Normalized' column:")
df_rice.head()

## Descriptive Statistics for Rice Yield Score


In [ ]:
print(df_rice['Yield_Score_Normalized'].describe())

## Correlation Matrix for Rice



In [ ]:
features_for_correlation = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'Yield_Score_Normalized']
correlation_matrix_rice = df_rice[features_for_correlation].corr()
print("Correlation Matrix for df_rice with Yield_Score_Normalized:")
print(correlation_matrix_rice)

## Scatter Plots for Rice


In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1) # 1 row, 2 columns, first plot
sns.scatterplot(data=df_rice, x='N', y='Yield_Score_Normalized')
plt.title('N vs. Normalized Yield Score for Rice')
plt.xlabel('N')
plt.ylabel('Normalized Yield Score')

plt.subplot(1, 2, 2) # 1 row, 2 columns, second plot
sns.scatterplot(data=df_rice, x='rainfall', y='Yield_Score_Normalized')
plt.title('Rainfall vs. Normalized Yield Score for Rice')
plt.xlabel('Rainfall')
plt.ylabel('Normalized Yield Score')

plt.tight_layout()
plt.show()

## Multiple Linear Regression for Rice

In [ ]:
import statsmodels.formula.api as smf

model_formula_rice = 'Yield_Score_Normalized ~ N + rainfall'
model_rice = smf.ols(formula=model_formula_rice, data=df_rice).fit()

print("OLS Model for Rice Yield Score (Yield_Score_Normalized ~ N + rainfall) built successfully.")

In [ ]:
print(model_rice.summary())

### Interpretation of Regression Results for Rice Yield

The OLS regression model `Yield_Score_Normalized ~ N + rainfall` for rice cultivation has been successfully built and its summary displayed. Let's interpret the key components:

1.  **R-squared Value:**
    *   The `R-squared` value is **0.994**. This indicates that approximately **99.4%** of the variance in the `Yield_Score_Normalized` for rice is explained by the independent variables `N` (Nitrogen) and `rainfall`. This vey  high R-squared value, suggesting that the model provides good fit to the data and that `N` and `rainfall` are very strong predictors of the  rice yield score.

2.  **Coefficient for 'N' (Nitrogen):**
    *   **Coefficient (coef):** The coefficient for `N` is **1.0624**. This means that for every one-unit increase in Nitrogen (`N`), the `Yield_Score_Normalized` is predicted to increase by approximately **1.0624 units**, assuming `rainfall` is held constant. The direction is positive, indicating a direct relationship: higher nitrogen levels are associated with higher yield scores.
    *   **P-value (P>|t|):** The p-value associated with `N` is **0.000**. Since this p-value is significantly less than the common alpha level of 0.05 (or even 0.01), the coefficient for `N` is **statistically highly significant**. This implies that the observed relationship between `N` and `Yield_Score_Normalized` is very unlikely to have occurred by chance.

3.  **Coefficient for 'rainfall':**
    *   **Coefficient (coef):** The coefficient for `rainfall` is **0.6422**. This implies that for every one-unit increase in `rainfall`, the `Yield_Score_Normalized` is predicted to increase by approximately **0.6422 units**, assuming `N` is held constant. The direction is positive, indicating a direct relationship: more rainfall is associated with higher yield scores.
    *   **P-value (P>|t|):** The p-value associated with `rainfall` is also **0.000**. Similar to `N`, this p-value is highly significant (p < 0.05), indicating that the effect of `rainfall` on `Yield_Score_Normalized` is **statistically highly significant** and not due to random chance.

4.  **Agronomic Implications:**
    *   Both nitrogen (`N`) and `rainfall` are shown to have a strong, positive, and statistically significant impact on the normalized rice yield score. The model suggests that increasing `N` levels and adequate `rainfall` are crucial for maximizing rice yield.
    *   The magnitude of the coefficients indicates that a change in `N` has a slightly larger per-unit impact on the yield score (1.0624) compared to `rainfall` (0.6422), within the specific scaling of these features in the model.
    *   These findings are consistent with general agricultural knowledge regarding rice cultivation. Rice is a heavy feeder of nitrogen, which is essential for vegetative growth, and it is a water-intensive crop, making rainfall a critical factor for its yield. The initial weighting in the `Yield_Score` engineering (0.5 for N, 0.3 for rainfall) also reflected this understanding, and the regression model largely supports these assumptions, particularly the strong positive impact of both factors.

## Multicollinearity for Rice

### Using:
Variance Inflation Factor (VIF) for the predictor variables ('N' and 'rainfall') in the rice regression model to assess potential multicollinearities.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

X_rice = df_rice[['N', 'rainfall']]
X_rice = sm.add_constant(X_rice)

vif_data_rice = pd.DataFrame()
vif_data_rice["feature"] = X_rice.columns
vif_data_rice["VIF"] = [variance_inflation_factor(X_rice.values, i) for i in range(len(X_rice.columns))]

print("Variance Inflation Factor (VIF) for Rice Model:")
print(vif_data_rice)

### Interpretation of VIF Results for Rice Model


1.  **VIF for 'N' (Nitrogen):** The VIF for 'N' is approximately **1.003**.
common rule of thumb : VIF values greater than 5 or 10 indicate potential multicollinearity issues.
Since the VIF for 'N' is very close to 1,
it indicates that 'N' has **no significant multicollinearity** with 'rainfall' in this model.

2.  **VIF for 'rainfall':** Similarly, the VIF for 'rainfall' is also approximately **1.003**. This value also suggests **no significant multicollinearity** between 'rainfall' and 'N'.

3.  **VIF for 'const' (Intercept):** The VIF for the constant term is high
(89.5). A high VIF for the constant is generally not a cause for concern regarding multicollinearity among the *predictor variables themselves*. It simply means that the mean of the predictors is far from zero, which is expected with real-world data and does not impact the coefficients or standard errors of the actual predictor variables ('N' and 'rainfall').

**Conclusion:**

Based on these VIF values, we can conclude that there is **no significant multicollinearity** between the independent variables 'N' and 'rainfall' in the regression model for predicting `Yield_Score_Normalized` for rice. This means that the effects of 'N' and 'rainfall' on the yield score can be reliably estimated independently by the model, and their standard errors are not inflated due to their inter-correlation.

## Plot Residual Distribution for Rice

plot (eX :a histogram or Q-Q plot) of the residuals from the rice regression model to assess their distribution and check for normality and homoscedasticity.


In [ ]:
residuals_rice = model_rice.resid

plt.figure(figsize=(8, 6))
sns.histplot(residuals_rice, kde=True)
plt.title('Distribution of Residuals for Rice Model')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
import statsmodels.api as sm

plt.figure(figsize=(8, 6))
sm.qqplot(residuals_rice, line='s')
plt.title('Q-Q Plot of Residuals for Rice Model')
plt.xlabel('Theoretical Quantiles')
plt.ylabel('Sample Quantiles')
plt.grid(True)
plt.show()